# state-est-testbed Live Demo
**Monte-Carlo Performance Analysis**

Linear Kalman Filter on a maneuvering target (sinusoidal lateral maneuver + oscillating lift)

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import stateest

print('stateest version:', getattr(stateest, '__version__', 'unknown'))

In [ ]:
# 2. Generate Trajectory (if not already present)
import subprocess
if not pd.io.common.file_exists('maneuver_kalman_test_data.csv'):
    print("Generating trajectory with simple.py...")
    subprocess.run(["python", "simple.py"], check=True)
else:
    print("Using existing maneuver_kalman_test_data.csv")

# 3. Load data
df = pd.read_csv('maneuver_kalman_test_data.csv')

print(f"Loaded {len(df)} timesteps")
print(f"Altitude range: {df['true_z_m'].min()/1000:.2f} – {df['true_z_m'].max()/1000:.2f} km")
print(f"Max lateral displacement: {df['true_y_m'].abs().max():.0f} m")

# Full 6D truth state for proper NEES calculation
x_true = np.hstack([
    df[['true_x_m', 'true_y_m', 'true_z_m']].values,
    df[['true_vx_mps', 'true_vy_mps', 'true_vz_mps']].values
])

In [ ]:
# 4. Kalman Filter Setup (Constant Velocity Model)
dt = df['time_s'].diff().mean()

F = np.eye(6)
F[0:3, 3:6] = dt * np.eye(3)

Q = np.diag([0., 0., 0., 0.5, 0.5, 0.5])
R = np.diag([15**2, 15**2, 15**2, 1.5**2, 0.1**2, 0.1**2])
H = np.eye(6)
B = np.zeros((6, 1))

print(f"F shape: {F.shape}")
print(f"Process noise on velocity: {np.diag(Q)[3:6]}")

In [ ]:
# 5. Monte-Carlo Simulation
n_mc = 100
rmse_pos = []
rmse_vel = []
nees_final = []

for run in tqdm(range(n_mc), desc="Monte-Carlo runs"):
    kf = stateest.KalmanFilter(F, Q, H, R, B)
    
    x0 = np.array([800000.0, 0.0, 80000.0, -5500.0, 0.0, -30.0])
    P0 = np.diag([1000**2]*3 + [200**2]*3)
    kf.init(x0, P0)
    
    estimates = []
    
    for i in range(len(df)):
        kf.predict()
        z = x_true[i] + np.random.randn(6) * np.sqrt(np.diag(R))
        kf.update(z)
        estimates.append(kf.state.copy())
    
    estimates = np.array(estimates)
    
    # Metrics
    rmse_pos.append(np.sqrt(np.mean((estimates[:, :3] - x_true[:, :3])**2)))
    rmse_vel.append(np.sqrt(np.mean((estimates[:, 3:] - x_true[:, 3:])**2)))
    
    final_err = estimates[-1] - x_true[-1]
    final_P = kf.covariance()
    final_nees = final_err @ np.linalg.solve(final_P, final_err)
    nees_final.append(final_nees)

print(f"\nMonte-Carlo completed ({n_mc} runs)")
print(f"Mean Position RMSE: {np.mean(rmse_pos):.1f} m")
print(f"Mean Velocity RMSE: {np.mean(rmse_vel):.2f} m/s")
print(f"Mean Final NEES   : {np.mean(nees_final):.2f} (expected ≈ 6.0)")

In [ ]:
# 6. Visualization
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

axs[0,0].hist(rmse_pos, bins=20, alpha=0.7, color='blue')
axs[0,0].set_title('Position RMSE Distribution')
axs[0,0].set_xlabel('RMSE [m]')

axs[0,1].hist(rmse_vel, bins=20, alpha=0.7, color='orange')
axs[0,1].set_title('Velocity RMSE Distribution')
axs[0,1].set_xlabel('RMSE [m/s]')

# Lateral maneuver tracking (last run)
axs[1,0].plot(df['time_s'], x_true[:,1], label='True Lateral (y)', lw=2)
axs[1,0].plot(df['time_s'], estimates[:,1], '--', label='Estimated y', lw=1.5)
axs[1,0].set_title('Lateral Maneuver Tracking')
axs[1,0].set_xlabel('Time [s]')
axs[1,0].legend()
axs[1,0].grid(True)

axs[1,1].plot(nees_final, 'o', alpha=0.6, markersize=4)
axs[1,1].axhline(6.0, color='red', ls='--', label='Expected NEES = 6')
axs[1,1].set_title('Final NEES per Run')
axs[1,1].set_xlabel('Monte-Carlo Run')
axs[1,1].legend()
axs[1,1].grid(True)

plt.tight_layout()
plt.show()

**Current Limitations & Next Steps**
- Using fake Cartesian measurements (truth + noise)
- Linear KF with nonlinear measurement model → will diverge in real scenarios
- Next: Implement proper EKF update using actual `meas_range_m`, `meas_az_rad`, `meas_el_rad`, `meas_rr_mps`